# medRAG Textbook Ingestion Pipeline (GPU Accelerated)

This notebook allows you to parse, chunk, embed, and ingest large medical textbooks using Google Colab or Kaggle GPUs. The processed data is stored directly in your cloud-hosted **Railway PostgreSQL database**.

### Why run this on Colab/Kaggle?
- **GPU Acceleration**: Docling and `sentence-transformers` use PyTorch. Colab/Kaggle GPUs (T4, L4, etc.) are 10-20x faster than local CPU parsing.
- **High RAM**: Medical textbooks (like Snell's Anatomy or Guyton's Physiology) are large and require significant memory during parsing.

## Step 1: Install Dependencies
First, select a GPU runtime (Colab: `Runtime` -> `Change runtime type` -> select `T4 GPU` or similar). Then run this cell to install all required libraries.

In [ ]:
!pip install docling sentence-transformers google-generativeai psycopg2-binary pgvector sqlalchemy pydantic-settings python-dotenv pypdfium2

## Step 2: Mount Google Drive or Create PDF Folder
If your textbooks are in Google Drive, run this cell to mount it. Otherwise, skip this and use the sidebar file upload to upload your PDFs into a folder named `pdfs`.

In [ ]:
import os
from google.colab import drive

try:
  drive.mount('/content/drive')
  print("Google Drive mounted.")
except Exception as e:
  print("Not running in Colab or failed to mount drive. Please upload your PDFs manually.")
  os.makedirs("/content/pdfs", exist_ok=True)
  print("Created folder: /content/pdfs")

## Step 3: Enter Database & API Credentials
Get your connection URL from your **Railway PostgreSQL service** and your **Gemini API Key**.

In [ ]:
import getpass
import os

DATABASE_URL = getpass.getpass("Enter Railway DATABASE_URL (postgresql://...): ")
GEMINI_API_KEY = getpass.getpass("Enter GEMINI_API_KEY (optional, press Enter to skip): ")

os.environ["DATABASE_URL"] = DATABASE_URL
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print("Credentials set in environment.")

## Step 4: Define Ingestion Logic
Run this cell to set up the DB models and ingestion pipeline functions.

In [ ]:
import io
import logging
from pathlib import Path
from datetime import datetime
from sqlalchemy import create_engine, text, ForeignKey, LargeBinary, Text, func, CheckConstraint
from sqlalchemy.dialects.postgresql import JSONB
from sqlalchemy.orm import declarative_base, mapped_column, Mapped, relationship, sessionmaker
from pgvector.sqlalchemy import Vector

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("medrag_colab")

# ================== DB Setup ==================
engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

class Book(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(Text)
    filename: Mapped[str] = mapped_column(Text, unique=True)
    status: Mapped[str] = mapped_column(Text, default="pending")
    error_message: Mapped[str | None] = mapped_column(Text, default=None)
    total_pages: Mapped[int | None] = mapped_column(default=None)
    created_at: Mapped[datetime] = mapped_column(server_default=func.now())
    updated_at: Mapped[datetime] = mapped_column(server_default=func.now(), onupdate=func.now())

class Chunk(Base):
    __tablename__ = "chunks"
    id: Mapped[int] = mapped_column(primary_key=True)
    book_id: Mapped[int] = mapped_column(ForeignKey("books.id", ondelete="CASCADE"))
    chapter: Mapped[str | None] = mapped_column(Text, default=None)
    page_number: Mapped[int | None] = mapped_column(default=None)
    content: Mapped[str] = mapped_column(Text)
    embedding: Mapped[list[float] | None] = mapped_column(Vector(1024), default=None)
    created_at: Mapped[datetime] = mapped_column(server_default=func.now())

class Figure(Base):
    __tablename__ = "figures"
    id: Mapped[int] = mapped_column(primary_key=True)
    book_id: Mapped[int] = mapped_column(ForeignKey("books.id", ondelete="CASCADE"))
    figure_label: Mapped[str | None] = mapped_column(Text, default=None)
    caption: Mapped[str | None] = mapped_column(Text, default=None)
    page_number: Mapped[int | None] = mapped_column(default=None)
    image_data: Mapped[bytes] = mapped_column(LargeBinary)
    mime_type: Mapped[str] = mapped_column(Text, default="image/png")
    created_at: Mapped[datetime] = mapped_column(server_default=func.now())

# ================== Ingestion pipeline logic ==================
def init_database():
    with engine.connect() as conn:
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
        conn.commit()
    Base.metadata.create_all(engine)
    logger.info("Database tables verified/created.")

def validate_pdf(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic != b"%PDF":
        raise ValueError(f"Not a valid PDF (magic bytes: {magic!r})")

def ingest_book(pdf_path: Path, title: str | None = None):
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.chunking import HierarchicalChunker
    from docling_core.types.doc import PictureItem
    from sentence_transformers import SentenceTransformer

    validate_pdf(pdf_path)
    if title is None:
        title = pdf_path.stem.replace("-", " ").replace("_", " ").title()

    session = SessionLocal()
    try:
        book = Book(title=title, filename=pdf_path.name, status="processing")
        session.add(book)
        session.commit()
        book_id = book.id
        logger.info(f"Starting ingestion: id={book_id}, title='{title}'")

        # 1. Parse PDF via Docling
        logger.info("Step 1: Parsing PDF...")
        pipeline_options = PdfPipelineOptions()
        pipeline_options.generate_picture_images = True
        pipeline_options.images_scale = 2.0
        converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})
        result = converter.convert(str(pdf_path))

        # 2. Extract Figures
        logger.info("Step 2: Extracting figures...")
        figures_data = []
        fig_count = 0
        doc = result.document
        for element, _level in doc.iterate_items():
            if isinstance(element, PictureItem):
                pil_image = element.get_image(doc)
                if pil_image:
                    fig_count += 1
                    buf = io.BytesIO()
                    pil_image.save(buf, format="PNG")
                    img_bytes = buf.getvalue()
                    page_num = element.prov[0].page_no if hasattr(element, "prov") and element.prov else None

                    figures_data.append({
                        "book_id": book_id,
                        "figure_label": f"Figure {fig_count}",
                        "caption": None,  # Captioned on-demand during query phase
                        "page_number": page_num,
                        "image_data": img_bytes,
                        "mime_type": "image/png"
                    })

        # 3. Chunking
        logger.info("Step 3: Chunking...")
        chunker = HierarchicalChunker(max_tokens=512)
        doc_chunks = list(chunker.chunk(doc))
        chunks_data = []
        for dc in doc_chunks:
            page_num = None
            if hasattr(dc, "meta") and dc.meta:
                prov = getattr(dc.meta, "doc_items", None)
                if prov:
                    for item in prov:
                        if hasattr(item, "prov") and item.prov:
                            page_num = item.prov[0].page_no
                            break
            chapter = " > ".join(dc.meta.headings) if hasattr(dc, "meta") and dc.meta and getattr(dc.meta, "headings", None) else None
            chunks_data.append({
                "book_id": book_id,
                "chapter": chapter,
                "page_number": page_num,
                "content": dc.text
            })

        # 4. Embeddings (runs on GPU if available!)
        logger.info("Step 4: Embedding chunks...")
        model = SentenceTransformer("BAAI/bge-large-en-v1.5")
        texts = [c["content"] for c in chunks_data]
        embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        for c, emb in zip(chunks_data, embeddings):
            c["embedding"] = emb.tolist()

        # 5. Save to database
        logger.info("Step 5: Saving database records...")
        for fig in figures_data:
            session.add(Figure(**fig))
        for chunk in chunks_data:
            session.add(Chunk(**chunk))
        
        book.status = "ready"
        book.total_pages = doc.num_pages() if hasattr(doc, "num_pages") else None
        session.commit()
        logger.info(f"Ingested successfully! {len(chunks_data)} chunks, {len(figures_data)} figures.")
        return book_id

    except Exception as e:
        session.rollback()
        book = session.get(Book, book_id) if 'book_id' in locals() else None
        if book:
            book.status = "failed"
            book.error_message = str(e)[:500]
            session.commit()
        logger.error(f"Failed ingesting: {e}")
        raise
    finally:
        session.close()

## Step 5: Start Ingestion!
Set the path below to your folder containing the PDFs (e.g. `"/content/drive/MyDrive/medRAG_pdfs"` or `"/content/pdfs"`), initialize the database tables, and run the batch processor.

In [ ]:
# Initialize database schema
init_database()

# Source folder containing textbooks
PDF_FOLDER = Path("/content/pdfs") # Adjust if using Google Drive

pdf_files = sorted(list(PDF_FOLDER.glob("*.pdf")))
print(f"Found {len(pdf_files)} PDF files in {PDF_FOLDER}:")
for f in pdf_files:
  print(f"  - {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

print("\nStarting batch ingestion...")
for idx, pdf in enumerate(pdf_files, 1):
  print(f"\n[{idx}/{len(pdf_files)}] Ingesting {pdf.name}...")
  try:
    ingest_book(pdf)
  except Exception as e:
    print(f"Error processing {pdf.name}: {e}")